# Medical Image Segmentation - Data Exploration

This notebook provides tools to explore and visualize medical imaging data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
from torch.utils.data import DataLoader

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (15, 10)

## 1. Load Data

In [ ]:
from utils.data_loader import MedicalImageDataset, get_preprocessing

# Replace with your actual data paths
image_dir = Path('data/images')
mask_dir = Path('data/masks')

image_paths = sorted(image_dir.glob('*.nii.gz'))
mask_paths = sorted(mask_dir.glob('*.nii.gz'))

print(f'Found {len(image_paths)} images and {len(mask_paths)} masks')

## 2. Visualize Sample Images

In [ ]:
# Create dataset
preprocessing = get_preprocessing('minmax')
dataset = MedicalImageDataset(image_paths[:10], mask_paths[:10], preprocessing=preprocessing, mode='val')

# Visualize samples
fig, axes = plt.subplots(5, 4, figsize=(15, 15))

for idx in range(min(5, len(dataset))):
    image, mask = dataset[idx]
    
    # First channel of image
    axes[idx, 0].imshow(image[0], cmap='gray')
    axes[idx, 0].set_title(f'Image {idx} - Channel 1')
    axes[idx, 0].axis('off')
    
    # Second channel
    axes[idx, 1].imshow(image[1], cmap='gray')
    axes[idx, 1].set_title(f'Image {idx} - Channel 2')
    axes[idx, 1].axis('off')
    
    # Third channel
    axes[idx, 2].imshow(image[2], cmap='gray')
    axes[idx, 2].set_title(f'Image {idx} - Channel 3')
    axes[idx, 2].axis('off')
    
    # Mask
    axes[idx, 3].imshow(mask, cmap='jet')
    axes[idx, 3].set_title(f'Mask {idx}')
    axes[idx, 3].axis('off')

plt.tight_layout()
plt.show()

## 3. Data Statistics

In [ ]:
# Calculate statistics
images_stats = []
masks_stats = {'class_distribution': {}}

for idx in range(len(dataset)):
    image, mask = dataset[idx]
    
    # Image statistics
    images_stats.append({
        'mean': image.mean().item(),
        'std': image.std().item(),
        'min': image.min().item(),
        'max': image.max().item(),
    })
    
    # Mask statistics
    unique_classes, counts = torch.unique(mask, return_counts=True)
    for cls, count in zip(unique_classes, counts):
        if cls.item() not in masks_stats['class_distribution']:
            masks_stats['class_distribution'][cls.item()] = 0
        masks_stats['class_distribution'][cls.item()] += count.item()

# Print statistics
print('Image Statistics:')
print(f'  Mean: {np.mean([s["mean"] for s in images_stats]):.4f}')
print(f'  Std:  {np.mean([s["std"] for s in images_stats]):.4f}')
print(f'  Min:  {np.mean([s["min"] for s in images_stats]):.4f}')
print(f'  Max:  {np.mean([s["max"] for s in images_stats]):.4f}')

print('\nMask Class Distribution:')
for cls, count in sorted(masks_stats['class_distribution'].items()):
    print(f'  Class {cls}: {count} pixels')

## 4. Class Distribution Visualization

In [ ]:
# Plot class distribution
classes = list(masks_stats['class_distribution'].keys())
counts = list(masks_stats['class_distribution'].values())

plt.figure(figsize=(10, 6))
plt.bar(classes, counts, color='steelblue')
plt.xlabel('Class')
plt.ylabel('Number of Pixels')
plt.title('Class Distribution in Masks')
plt.xticks(classes)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()